In [1]:
!pip install -q transformers requests sentence-transformers
import os, getpass
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

HF token (read scope): ··········


In [4]:
import os
import time
import requests
from requests.exceptions import RequestException

# =====================================================
# CONFIG
# =====================================================

USE_API = True   # Set False to force local model

HF_TOKEN = os.environ.get("HF_TOKEN", "")

API_URL = (
    "https://api-inference.huggingface.co/models/"
    "facebook/bart-large-mnli"
)

HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

# =====================================================
# TEST DATA
# =====================================================

resumes = [
    "Built React dashboards for 3 startups",
    "Implemented Spring Boot microservices in Java for fintech app",
    "Trained CNN for image classification using PyTorch, 87% accuracy",
    "Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports",
    "Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x",
]

labels = [
    "frontend dev",
    "backend dev",
    "data analyst",
    "ML engineer",
]

# =====================================================
# API VERSION
# =====================================================

def classify_with_api(texts, labels, retries=3):

    payload = {
        "inputs": texts,
        "parameters": {
            "candidate_labels": labels
        }
    }

    for attempt in range(retries):

        try:
            response = requests.post(
                API_URL,
                headers=HEADERS,
                json=payload,
                timeout=60
            )

            result = response.json()

            # Handle HF loading
            if isinstance(result, dict) and "error" in result:

                err = result["error"]

                if "loading" in err.lower():
                    wait = result.get("estimated_time", 10)
                    print(f"Model loading... waiting {wait:.1f}s")
                    time.sleep(wait)
                    continue

                raise RuntimeError(err)

            return result

        except RequestException as e:
            print(f"API request failed: {e}")
            time.sleep(2)

    return None

# =====================================================
# LOCAL FALLBACK
# =====================================================

def classify_local(texts, labels):

    print("\nUsing LOCAL transformers model...\n")

    from transformers import pipeline

    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    outputs = []

    for text in texts:
        result = classifier(text, labels)
        outputs.append(result)

    return outputs

# =====================================================
# MAIN
# =====================================================

start = time.time()

results = None

# Try API first
if USE_API:

    print("Trying Hugging Face API...\n")

    results = classify_with_api(resumes, labels)

# Fallback to local model
if results is None:

    print("Falling back to local inference...")

    results = classify_local(resumes, labels)

# =====================================================
# DISPLAY RESULTS
# =====================================================

print("\nPredictions:\n")

for text, result in zip(resumes, results):

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    print(
        f"{text[:65]:65} "
        f"-> {top_label:15} "
        f"({top_score:.2f})"
    )

print(f"\nTotal time: {time.time() - start:.2f}s")

Trying Hugging Face API...

API request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x787719ecf290>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
API request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x787719ece030>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
API request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x787719ecd8b0>: Failed to resolve 'api-inference.huggingface.co' 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Predictions:

Built React dashboards for 3 startups                             -> frontend dev    (0.88)
Implemented Spring Boot microservices in Java for fintech app     -> backend dev     (0.63)
Trained CNN for image classification using PyTorch, 87% accuracy  -> ML engineer     (0.40)
Cleaned 100k row dataset using pandas + plotted in seaborn for mo -> data analyst    (0.56)
Wrote SQL queries against PostgreSQL, optimised 3 slow queries by -> data analyst    (0.45)

Total time: 64.12s


In [5]:
from transformers import pipeline

# This DOWNLOADS the model on first run — ~1.6GB. Be patient.
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

  Built React dashboards for 3 startups              -> frontend dev (0.88)
  Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
  Trained CNN for image classification using PyTorch -> ML engineer (0.40)
  Cleaned 100k row dataset using pandas + plotted in -> data analyst (0.56)
  Wrote SQL queries against PostgreSQL, optimised 3  -> data analyst (0.45)

Local time (after download): 12.25s


In [6]:
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')

# Test data — 5 mock interview answers
answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment scores:
  [POSITIVE 1.00] I really enjoyed working on the team and shipped 3 features.
  [NEGATIVE 1.00] I was the only one writing code; everyone else was slow.
  [POSITIVE 1.00] I learned a lot from my mentor and grew technically.
  [NEGATIVE 1.00] I had to redo most of my teammate's work because it was wron
  [POSITIVE 1.00] My internship was great — would recommend it to anyone.


In [10]:
import time

# =====================================================
# Timing helper
# =====================================================

def time_call(fn, n_runs=3):

    times = []

    for _ in range(n_runs):

        start = time.time()
        fn()
        times.append(time.time() - start)

    return min(times), sum(times) / len(times)

# =====================================================
# API timing
# =====================================================

api_min = None
api_avg = None

def call_api():

    hf_zero_shot_api(
        'Built React dashboards',
        ['frontend dev', 'backend dev']
    )

try:

    api_min, api_avg = time_call(call_api)

except Exception as e:

    print(f'API timing skipped: {e}')

# =====================================================
# Local timing
# =====================================================

def call_local():

    classifier(
        'Built React dashboards',
        candidate_labels=[
            'frontend dev',
            'backend dev'
        ]
    )

local_min, local_avg = time_call(call_local)

# =====================================================
# Results table
# =====================================================

print('\nInference timing comparison (3 runs each, after warm-up):\n')

if api_min is not None:

    print(f'API:   min {api_min:.2f}s | avg {api_avg:.2f}s')

else:

    print('API:   unavailable (network/DNS failure)')

print(f'Local: min {local_min:.2f}s | avg {local_avg:.2f}s')

Request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7876bfa26c00>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
Request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7876bfa265d0>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)"))
Request failed: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/facebook/bart-large-mnli (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7876bfa25ca0>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)")